# 01 · pandas basics

**Session 3 — Reproducible Foundations** · about 45 minutes

**You will learn to**
- load tables from CSV, Excel and Parquet files
- work with Series and DataFrames, select with `loc` / `iloc` and filter with boolean masks
- recognise pandas 3 behaviour (Copy-on-Write, the `str` dtype) and outdated AI-written code
- clean messy columns and profile a dataset with pandas alone

**Data:** sleep in adolescent mice (Nelson et al. 2013), IMDB top 1000 (Kaggle), electricity consumption by municipality (Transparència Catalunya). Sources and licences: `data/README.md`.

**Outline**
0. Before you start (no AI)
1. Loading data
2. Series
3. DataFrames
4. pandas 3: Copy-on-Write and outdated code
5. Selecting and filtering
6. Cleaning messy columns
7. Profiling with pandas
8. Your dataset

> **AI guardrails** — *Think first · Own it · Verify · Disclose · Protect.* AI tools are allowed in work time, but answer the **No-AI** questions on paper first, and make every ✅ self-check pass with code you can explain line by line.

## 0. Before you start (no AI)

Write your answers on paper, then keep reading: the notebook answers them.

1. A table has 5 rows with index labels `10, 20, 30, 40, 50`. What do `df.loc[20]` and `df.iloc[2]` return?
2. What is the difference between a *Series* and a *DataFrame*?
3. In your project dataset, **what is one row**?

## 1. Loading data

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print("pandas", pd.__version__)

In [ ]:
from pathlib import Path

DATA = Path("../data")
OUTPUTS = Path("../outputs")
OUTPUTS.mkdir(exist_ok=True)
assert DATA.exists(), "Start Jupyter from the repository folder: notebooks/ must sit next to data/."

Files are read with `pd.read_<format>()` and written with `df.to_<format>()`.
Data: Nelson AB et al. *Sleep patterns and homeostatic mechanisms in adolescent mice.* Brain Sci. 2013;3(1):318–43.

In [ ]:
mice_csv = pd.read_csv(DATA / "Sleep_Patterns_and_Homeostatic_Mechanisms_in_Adolescent_Mice_2013.csv",
                       index_col=0)
mice_xlsx = pd.read_excel(DATA / "Sleep_Patterns_and_Homeostatic_Mechanisms_in_Adolescent_Mice_2013.xlsx",
                          index_col=0)
mice_csv.head()

**Parquet** is the format we use for intermediate data: it keeps column types, is compressed and is fast.
This file was downloaded from the open data portal (notebook `03_open_data_apis` shows how).

In [ ]:
energy = pd.read_parquet(DATA / "examples/energy_consumption_municipalities.parquet")
print(energy.shape)
energy.head()

## 2. Series: one labelled column

In [ ]:
observed_birds = pd.Series(
    ["Wallcreeper", "Lammergeier", "Lammergeier", "Lammergeier", "Dupont's Lark",
     "Pin-tailed Sandgrouse", "Lammergeier", "Wallcreeper", "Little Bustard", "Dupont's Lark",
     "Wallcreeper", "Wallcreeper", "Citril Finch", "Black Wheatear", "Snowfinch"],
    name="birds",
)
counts = observed_birds.value_counts()
ax = counts.sort_values().plot.barh(color="tab:blue")
ax.set_xlabel("Number of observations")
plt.show()

A bar chart, not a pie chart: lengths are easier to compare than angles (Session 10).

Series hold numbers too. Below: 10,000 draws from a Gaussian with mean 2 and standard deviation 1.
`default_rng(42)` fixes the random seed so everybody gets the same numbers.

In [ ]:
rng = np.random.default_rng(42)
mu, sigma = 2, 1
samples = pd.Series(rng.normal(mu, sigma, 10_000), name="x")
samples.describe()

### TASK I — make the histogram tell the truth

1. Plot a **normalised** histogram of `samples` with 50 bins (`density=True`).
2. Overlay the analytical Gaussian density on the same axes.
3. Label both axes, add a title and a legend.
4. Compute `within_1sigma`: the fraction of samples between μ − σ and μ + σ.

*Predict first:* what value should `within_1sigma` be close to?

In [ ]:
# TASK I — your code here
fig, ax = plt.subplots()

within_1sigma = None

In [ ]:
# ✅ Self-check TASK I
assert ax.get_xlabel() and ax.get_ylabel(), "Label both axes."
assert ax.get_legend() is not None, "Add a legend."
assert within_1sigma is not None and 0.67 < within_1sigma < 0.70, "Expected about 0.683 (68% rule)."
print("✅ TASK I passed")

## 3. DataFrames: tables of Series sharing an index

In [ ]:
countries = pd.DataFrame({
    "country": ["UK", "France", "Spain"],
    "capital": ["London", "Paris", "Madrid"],
    "pop_millions": [67.22, 67.39, 47.35],
})
countries

In [ ]:
print(countries.columns.tolist())
print(countries.index)
print(countries.dtypes)

Column names without spaces let you write `countries.capital`; `countries["capital"]` always works.
`set_index` / `reset_index` move a column into the index and back.

In [ ]:
by_country = countries.set_index("country")
by_country.loc["Spain", "capital"]

## 4. pandas 3: Copy-on-Write and outdated code

pandas 3.0 (January 2026) changed two things you will meet every day:

- **Copy-on-Write**: a selection behaves like a copy. Modifying it never changes the original, and
  *chained assignment* (`df[mask]["col"] = value`) silently does nothing (you only get a warning).
- **Strings** have their own dtype, `str`, instead of the generic `object`.

AI assistants learned from years of pandas 1.x code, so their suggestions often use removed or
changed features. Run the next cell: it was written by an AI assistant.

### Verify the AI

In [ ]:
# An AI assistant suggested this to add a row. Run it and read the error.
countries.append({"country": "Germany", "capital": "Berlin", "pop_millions": 83.24}, ignore_index=True)

`DataFrame.append` was removed in pandas 2.0. The replacement is `pd.concat`:

In [ ]:
germany = pd.DataFrame([{"country": "Germany", "capital": "Berlin", "pop_millions": 83.24}])
pd.concat([countries, germany], ignore_index=True)

Chained assignment does nothing in pandas 3. Compare the two ways of writing the same intention:

In [ ]:
import warnings

demo = countries.copy()
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    demo[demo["pop_millions"] > 50]["capital"] = "BIG"        # chained: no effect
print("after chained assignment:", demo["capital"].tolist(), "| warning:",
      [type(w.message).__name__ for w in caught])

demo.loc[demo["pop_millions"] > 50, "capital"] = "BIG"        # one .loc call: works
print("after .loc assignment:   ", demo["capital"].tolist())

### TASK II — update the table, the pandas 3 way

Starting from `countries`, build `updated` that:

1. adds Germany (Berlin, 83.24 million) with `pd.concat`;
2. adds a boolean column `eu_member`;
3. updates France's population to 68.0 with `.loc`;
4. is sorted by population, largest first;
5. is saved to `OUTPUTS / "updated_countries.csv"` without the index.

In [ ]:
# TASK II — your code here
updated = None

In [ ]:
# ✅ Self-check TASK II
assert updated is not None and len(updated) == 4, "Four countries expected."
assert updated["pop_millions"].is_monotonic_decreasing, "Sort by population, largest first."
assert updated.set_index("country").loc["France", "pop_millions"] == 68.0
assert updated.set_index("country")["eu_member"].to_dict() == {
    "Germany": True, "France": True, "UK": False, "Spain": True}
assert (OUTPUTS / "updated_countries.csv").exists()
print("✅ TASK II passed")

## 5. Selecting and filtering

IMDB top 1000 films ([Kaggle](https://www.kaggle.com/datasets/omarhanyy/imdb-top-1000)).

In [ ]:
imdb = pd.read_csv(DATA / "imdb/imdb_top_1000.csv")
imdb.head(3)

- `.loc[row_label, column_label]` selects by **label**; `.iloc[i, j]` by **position** (end excluded).
- A boolean **mask** keeps the rows where it is `True`; combine masks with `&`, `|` and parentheses.

In [ ]:
print(imdb.loc[2, "Series_Title"])
print(imdb.iloc[1:3, 1].tolist())
imdb[(imdb["Director"] == "Christopher Nolan") & (imdb["IMDB_Rating"] >= 8.5)][["Series_Title", "IMDB_Rating"]]

`Released_Year` should be a number, but one row says `"PG"`. `pd.to_numeric(..., errors="coerce")`
turns anything that is not a number into a missing value, so you can find it:

In [ ]:
imdb["year"] = pd.to_numeric(imdb["Released_Year"], errors="coerce").astype("Int64")
print(imdb.loc[imdb["year"].isna(), ["Series_Title", "Released_Year"]])
imdb[imdb["year"] >= 2019][["Series_Title", "year", "IMDB_Rating"]]

## 6. Cleaning messy columns

Real columns rarely arrive ready to compute with: `Runtime` is text like `"142 min"`,
`Gross` is text like `"28,341,469"` and `Genre` packs several genres into one string.

### TASK III — clean three columns

Add to `imdb`:

1. `runtime_min` — runtime in minutes as an integer;
2. `gross_usd` — gross revenue as a float (missing where unknown);
3. a Series `top_genres` with the 5 most frequent genres, counting every genre of every film
   (hint: `str.split(", ")` and `explode`).

*Predict first:* which genre do you expect at the top?

In [ ]:
# TASK III — your code here
top_genres = None

In [ ]:
# ✅ Self-check TASK III
assert imdb["runtime_min"].dtype.kind == "i" and imdb["runtime_min"].max() == 321
assert imdb["gross_usd"].notna().sum() == 831, "831 films have a known gross."
assert top_genres is not None and top_genres.index[0] == "Drama" and top_genres.iloc[0] == 724
print("✅ TASK III passed")

## 7. Profiling with pandas

Before any analysis, look at the whole table: types, missing values, distinct values, ranges.
pandas does this without extra libraries.

In [ ]:
imdb.info()

In [ ]:
imdb.describe(include="all").T.head(8)

In [ ]:
imdb.isna().mean().sort_values(ascending=False).head()

### TASK IV — your own `profile()` function

Write `profile(df)` returning one row per column of `df` with the columns
`dtype`, `n_missing`, `pct_missing` (0–100) and `n_unique`. You will reuse it on your dataset.

In [ ]:
# TASK IV — your code here
def profile(df):
    ...

In [ ]:
# ✅ Self-check TASK IV
p = profile(imdb)
assert p is not None and len(p) == imdb.shape[1], "One row per column."
assert {"dtype", "n_missing", "pct_missing", "n_unique"} <= set(p.columns)
assert p.loc["Meta_score", "n_missing"] == 157
print("✅ TASK IV passed")

## 8. Your dataset

In a new notebook in your group repository:

1. Load your dataset (file or API, see `03_open_data_apis`) and save a Parquet copy.
2. Run `profile()` on it.
3. Write a **dataset card** in a markdown cell:

| Field | Your answer |
|---|---|
| Source (URL) and licence | |
| Rows × columns | |
| What is one row? | |
| Columns you will use | |
| Missing values / problems found | |
| API access? | |

**Next:** `02_pandas_wrangling.ipynb` (Session 4).